# Classificação de imagens de paisagem com CNN

Rede neural convolucional (TensorFlow/Keras) que classifica imagens em 4 categorias:
**Rural**, **Urbano**, **Água** e **Área Verde**.

Para executar: menu **Ambiente de execução > Executar tudo**. Nada precisa ser instalado.

## 1. Baixar o projeto (imagens de treino e validação)

In [ ]:
import os

if not os.path.exists('aprendizagem'):
    !git clone https://github.com/Serg0-Proxy/aprendizagem.git

%cd aprendizagem

## 2. Importações e parâmetros

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping

train_dir = './Treinamento'
validation_dir = './Validacao'

input_shape = (150, 150, 3)
num_classes = 4
batch_size = 32
epochs = 30  # o early stopping interrompe antes, se necessário

## 3. Modelo

In [ ]:
model = Sequential([
    Input(shape=input_shape),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax'),
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## 4. Dados

O treino usa *data augmentation* (rotação, deslocamento, zoom e espelhamento). A validação só é normalizada.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='categorical'
)

validation_datagen = ImageDataGenerator(rescale=1./255)
validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='categorical'
)

## 5. Treinamento

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
    callbacks=[early_stop],
)

## 6. Avaliação

In [ ]:
loss, accuracy = model.evaluate(validation_generator)
print(f'Acurácia no conjunto de validação: {accuracy * 100:.2f}%')
print('Classes:', train_generator.class_indices)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='treino')
axes[0].plot(history.history['val_accuracy'], label='validação')
axes[0].set_title('Acurácia')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(history.history['loss'], label='treino')
axes[1].plot(history.history['val_loss'], label='validação')
axes[1].set_title('Perda (loss)')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.show()

## 7. Teste com sua própria imagem

Execute a célula abaixo, escolha uma imagem do seu computador (foto de uma cidade, rio, plantação, floresta...) e veja a previsão do modelo.

In [ ]:
from google.colab import files

uploaded = files.upload()
class_names = {v: k for k, v in train_generator.class_indices.items()}

for nome in uploaded:
    img = load_img(nome, target_size=(150, 150))
    x = img_to_array(img) / 255.0
    x = np.expand_dims(x, axis=0)

    probs = model.predict(x)[0]
    idx = int(np.argmax(probs))

    plt.imshow(img)
    plt.axis('off')
    plt.title(f'{class_names[idx]} ({probs[idx] * 100:.1f}%)')
    plt.show()

    for i, p in enumerate(probs):
        print(f'{class_names[i]}: {p * 100:.1f}%')